# 07 · Árboles binarios: recursión y `foldTree`

> **En el examen:** aparecen en el Ejercicio 3 o 4 en la mayoría de cursos (19-20, 21-22,
> 22-23, 23-24, 24-25). Te piden una función sobre árboles **recursiva** y luego **con
> `foldTree`** (o con `map` sobre árboles), o bien abstraer un patrón sobre árboles
> (`traverse`, `unfoldTree`, archivo 08). 3-5 puntos.

Requisito: archivo 04 (es la misma idea que en listas, pero con **dos** subproblemas).

---

### Preámbulo
Ejecuta la siguiente celda antes que las demás. Las celdas de código se pueden ejecutar en orden; los fragmentos que no son ejecutables por sí solos se muestran como texto.

In [ ]:
// Preámbulo: ejecuta esta celda primero (las mismas definiciones que vienen en los exámenes)
import $ivy.`org.scalatest::scalatest:3.2.16`
import org.scalatest.{Filter => _, _}, flatspec._, matchers._

type Not[P] = P => Nothing
type <=>[P, Q] = (P => Q, Q => P)

trait ExcludedMiddle:
    def apply[P]: Either[P, Not[P]]

trait DoubleNegation:
    def apply[P]: Not[Not[P]] => P

trait Isomorphic[A, B]:
    def from(a: A): B
    def to(b: B): A
    def equalA(a1: A, a2: A): Boolean = a1 == a2
    def equalB(b1: B, b2: B): Boolean = b1 == b2
    def law1(a: A): Boolean = equalA(to(from(a)), a)
    def law2(b: B): Boolean = equalB(from(to(b)), b)

enum Tree[A]:
    case Empty()
    case Node(left: Tree[A], root: A, right: Tree[A])

object Tree:
    def void[A]: Tree[A] = Empty()
    def leaf[A](a: A): Node[A] = Node(Empty(), a, Empty())
    def right[A](a: A, tree: Tree[A]): Node[A] = Node(Empty(), a, tree)
    def left[A](tree: Tree[A], a: A): Node[A] = Node(tree, a, Empty())
    def node[A](left: Tree[A], a: A, right: Tree[A]): Node[A] = Node(left, a, right)
    def foldTree[A, B](tree: Tree[A])(empty: B)(node: (B, A, B) => B): B =
        tree match
            case Empty() => empty
            case Node(left, a, right) => node(foldTree(left)(empty)(node), a, foldTree(right)(empty)(node))

import Tree._

def pow(base: Int, exp: Int): Int = math.pow(base, exp).toInt

## 1. El tipo `Tree`

En el preámbulo del examen:

In [ ]:
// type Tree[A] = 1 + Tree[A] * A * Tree[A]

enum Tree[A]:
    case Empty()
    case Node(left: Tree[A], root: A, right: Tree[A])

object Tree:
    def void[A]: Tree[A] = Empty()
    def leaf[A](a: A): Node[A] = Node(Empty(), a, Empty())
    def right[A](a: A, tree: Tree[A]): Node[A] = Node(Empty(), a, tree)   // solo hijo DERECHO
    def left[A](tree: Tree[A], a: A): Node[A] = Node(tree, a, Empty())    // solo hijo IZQUIERDO
    def node[A](left: Tree[A], a: A, right: Tree[A]): Node[A] = Node(left, a, right)

import Tree._

Un árbol es **o** vacío (`Empty()`) **o** un nodo con: subárbol izquierdo, valor de la raíz,
subárbol derecho.

Las funciones `void`, `leaf`, `left`, `right`, `node` son **atajos** para escribir árboles en
los tests. Fíjate en el orden de sus argumentos, que **imita el dibujo**: en `left(t, a)` el
hijo va a la izquierda de la raíz; en `right(a, t)` va a la derecha.

### Leer un árbol de un test

```scala
node(left(leaf(4), 1), 0, node(leaf(3), 2, right(2, right(4, leaf(5)))))
```

```
            0
          /   \
         1     2
        /     / \
       4     3   2
                  \
                   4
                    \
                     5
```

Dibújalo siempre en papel antes de pensar.

---

## 2. Recursión sobre árboles: divide y vencerás de verdad

En listas había un subproblema (`tail`). En árboles hay **dos**: el izquierdo y el derecho.

In [ ]:
def f[A, B](t: Tree[A]): B =
    t match
        case Empty() => ???                  // árbol vacío: solución directa
        case Node(left, root, right) =>
            val leftSol  = f(left)           // supón resuelto el izquierdo
            val rightSol = f(right)          // supón resuelto el derecho
            ??? /* combinar leftSol, root, rightSol */

### Ejemplos básicos (practica con estos primero)

In [ ]:
def size[A](t: Tree[A]): Int =
    t match
        case Empty() => 0
        case Node(l, _, r) => size(l) + 1 + size(r)

def sum(t: Tree[Int]): Int =
    t match
        case Empty() => 0
        case Node(l, x, r) => sum(l) + x + sum(r)

def depth[A](t: Tree[A]): Int =
    t match
        case Empty() => 0
        case Node(l, _, r) => 1 + math.max(depth(l), depth(r))

def inorder[A](t: Tree[A]): List[A] =             // recorrido en orden
    t match
        case Empty() => Nil
        case Node(l, x, r) => inorder(l) ++ (x :: inorder(r))

---

## 3. `foldTree`: la recursión sobre árboles ya escrita

En el preámbulo de varios exámenes:

In [ ]:
def foldTree[A, B](tree: Tree[A])(empty: B)(node: (B, A, B) => B): B =
    tree match
        case Empty() => empty
        case Node(left, a, right) =>
            node(foldTree(left)(empty)(node), a, foldTree(right)(empty)(node))

Igual que `foldRight`, pero la función de combinar recibe **tres** cosas:
`(solucionIzquierda, raiz, solucionDerecha)`.

### Receta: de recursión a `foldTree`

| Recursión | `foldTree` |
|---|---|
| `case Empty() => X` | `empty = X` |
| `case Node(l, a, r) => ... f(l) ... f(r) ...` | `(leftSol, a, rightSol) => ...` cambiando `f(l)` por `leftSol` y `f(r)` por `rightSol` |

In [ ]:
def size[A](t: Tree[A]): Int   = foldTree(t)(0)((l, _, r) => l + 1 + r)
def sum(t: Tree[Int]): Int     = foldTree(t)(0)((l, x, r) => l + x + r)
def depth[A](t: Tree[A]): Int  = foldTree(t)(0)((l, _, r) => 1 + math.max(l, r))

---

## 4. `map` sobre árboles (22-23 Extraordinaria)

Aplicar una función a cada nodo, manteniendo la forma:

In [ ]:
def map[A, B](t: Tree[A])(f: A => B): Tree[B] =
    t match
        case Empty() => Empty()
        case Node(left, root, right) => Node(map(left)(f), f(root), map(right)(f))

// con foldTree:
def mapFT[A, B](t: Tree[A])(f: A => B): Tree[B] =
    foldTree[A, Tree[B]](t)(Empty())((leftSol, root, rightSol) => Node(leftSol, f(root), rightSol))

(En `foldTree[A, Tree[B]]` decimos los tipos para que `Empty()` sepa que es un `Tree[B]`.)

Con `map`, muchas funciones son una línea:

In [ ]:
def duplicate[A](t: Tree[A]): Tree[(A, A)] = map(t)(a => (a, a))
def fusion(t: Tree[(Int, Int, Int)]): Tree[Int] = map(t) { case (a, b, c) => a + b + c }
def positives(t: Tree[Int]): Tree[Boolean] = map(t)(_ > 0)
def chars(t: Tree[String]): Tree[List[Char]] = map(t)(_.toList)

---

## 5. Ejemplos resueltos de exámenes

### Rama de coste máximo (21-22 Móstoles)

Rama = camino de la raíz a una hoja; coste = suma de sus nodos.

In [ ]:
def sum(list: List[Int]): Int = list.foldRight(0)(_ + _)

def highestCostBranch(tree: Tree[Int]): List[Int] =
    tree match
        case Empty() => List()
        case Node(left, root, right) =>
            val bestLeft  = highestCostBranch(left)
            val bestRight = highestCostBranch(right)
            root :: (if sum(bestRight) > sum(bestLeft) then bestRight else bestLeft)

Con `foldTree` **sin** usar `sum`: que la solución sea un **par** `(coste, rama)`, así no hay
que recalcular la suma (mismo truco de la tupla que en el archivo 04):

In [ ]:
def highestCostBranchFT(tree: Tree[Int]): List[Int] =
    foldTree(tree)((0, List[Int]())):
        case ((sumL, bestL), root, (sumR, bestR)) =>
            if sumL > sumR then (sumL + root, root :: bestL)
            else (sumR + root, root :: bestR)
    ._2

### Camino más largo (21-22 Vicálvaro)

Idéntico, cambiando "suma" por "longitud":

In [ ]:
def longestPathFT[A](tree: Tree[A]): List[A] =
    foldTree(tree)((0, List[A]())):
        case ((l, pathL), root, (r, pathR)) =>
            if l > r then (l + 1, root :: pathL) else (r + 1, root :: pathR)
    ._2

### ¿Está balanceado? (19-20 Prueba)

Balanceado: los dos hijos lo están y sus profundidades difieren como mucho en 1. La
solución lleva `(balanceado, profundidad)`:

In [ ]:
def balanced[A](tree: Tree[A]): Boolean =
    foldTree(tree)((true, 0)):
        case ((balL, depL), _, (balR, depR)) =>
            (balL && balR && (depL - depR).abs <= 1, 1 + math.max(depL, depR))
    ._1

### Nodos de un nivel (`breadth`, 22-23 Vicálvaro)

`breadth(tree)(n)` = lista de los nodos que están en el nivel `n` (la raíz es el 0).

In [ ]:
def breadth[A](tree: Tree[A])(level: Int): List[A] =
    tree match
        case Empty() => List()
        case Node(_, root, _) if level == 0 => List(root)
        case Node(left, _, right) => breadth(left)(level - 1) ++ breadth(right)(level - 1)

Con `foldTree` la "solución" de cada subárbol es **una función** `Int => List[A]` (la que
dado un nivel devuelve sus nodos):

In [ ]:
def breadthFT[A](tree: Tree[A]): Int => List[A] =
    foldTree(tree)((_: Int) => List[A]())(
        (breadthL, root, breadthR) =>
            level => if level == 0 then List(root)
                     else breadthL(level - 1) ++ breadthR(level - 1)
    )

### Árbol binario de búsqueda: insertar (19-20 Diciembre)

En un árbol de búsqueda, todo lo de la izquierda es menor que la raíz y todo lo de la
derecha es mayor (sin repetidos).

In [ ]:
def add(tree: Tree[Int], i: Int): Tree[Int] =
    tree match
        case Empty() => leaf(i)
        case Node(left, root, right) =>
            if i == root then tree                         // ya está
            else if i < root then Node(add(left, i), root, right)
            else Node(left, root, add(right, i))

def toTree(list: List[Int]): Tree[Int] =
    list.foldLeft(void[Int])(add)

### Árbol de búsqueda: quitar el mínimo (19-20 Julio)

El mínimo es el nodo más a la izquierda. Devuelve el árbol sin él y el propio mínimo.

In [ ]:
def removeMin[A](tree: Tree[A]): Option[(Tree[A], A)] =
    tree match
        case Empty() => None
        case Node(left, a, right) =>
            removeMin(left) match
                case None => Some((right, a))                     // no hay nada a la izquierda: a es el mínimo
                case Some((leftR, min)) => Some((Node(leftR, a, right), min))

Y con eso, borrar un elemento cualquiera: si lo encuentras, lo sustituyes por el mínimo de
su subárbol derecho.

In [ ]:
def remove(tree: Tree[Int], i: Int): Tree[Int] =
    tree match
        case Empty() => Empty()
        case Node(left, `i`, right) =>
            removeMin(right) match
                case None => left
                case Some((rightR, min)) => Node(left, min, rightR)
        case Node(left, a, right) if a < i => Node(left, a, remove(right, i))
        case Node(left, a, right)          => Node(remove(left, i), a, right)

### Otra definición de árbol: `Leaf` / `Branch` (22-23 Vicálvaro Especial)

A veces te dan otro árbol, con valores solo en las hojas:

In [ ]:
sealed trait Tree[A]
case class Leaf[A](a: A) extends Tree[A]
case class Branch[A](l: Tree[A], r: Tree[A]) extends Tree[A]

La receta es la misma: un caso por cada forma. Su `fold`:

In [ ]:
def fold[A, B](t: Tree[A])(leaf: A => B)(branch: (B, B) => B): B =
    t match
        case Leaf(a) => leaf(a)
        case Branch(l, r) => branch(fold(l)(leaf)(branch), fold(r)(leaf)(branch))

def total(t: Tree[String]): Int = fold(t)(_.length)(_ + _)      // suma de longitudes

**Regla general para cualquier tipo recursivo:** su `fold` recibe **una función por cada
forma** (`case`) del tipo, y en las partes recursivas recibe ya la solución.

---

## 6. Errores típicos

- Confundir el orden de `left(árbol, valor)` y `right(valor, árbol)`.
- En `foldTree`, olvidar que la función recibe **tres** argumentos `(izq, raíz, der)`.
- No poner tipo al valor inicial: `(0, List[Int]())` y no `(0, Nil)`.
- Árboles de búsqueda: olvidar el caso "ya está" (`i == root`).

---

## 7. Ejercicios para practicar

### De exámenes reales

| Función | Examen |
|---|---|
| `genAST`, `genTree`, `unfoldTree` | 24-25 Móstoles (archivo 08) |
| `heads`, `divide`, `traverse`, `sequence` | 23-24 Vicálvaro (archivo 08) |
| `duplicate`, `fusion`, `positives`, `chars`, `map` | 22-23 Extraordinaria |
| `breadth` | 22-23 Vicálvaro |
| `isEven`, `total`, `fold` (Leaf/Branch) | 22-23 Vicálvaro Especial |
| `highestCostBranch` | 21-22 Móstoles |
| `longestPath` | 21-22 Vicálvaro |
| `add`, `toTree` (BST) | 19-20 Diciembre |
| `remove` (8 variantes) | 19-20 Julio |
| `balanced` | 19-20 Prueba |

### Ejercicios nuevos (recursivo y con `foldTree`)

1. `leaves[A](t: Tree[A]): Int` — número de hojas (nodos sin hijos).
2. `maxTree(t: Tree[Int]): Option[Int]` — el mayor valor.
3. `mirror[A](t: Tree[A]): Tree[A]` — intercambia izquierda y derecha en todo el árbol.
4. `contains[A](t: Tree[A], x: A): Boolean`.

<details>
<summary>Soluciones con foldTree</summary>

```scala
def leaves[A](t: Tree[A]): Int =
    t match
        case Empty() => 0
        case Node(Empty(), _, Empty()) => 1
        case Node(l, _, r) => leaves(l) + leaves(r)
// con foldTree no se ve si un hijo es vacío; un truco es devolver (hojas, ¿es vacío?)
def leavesFT[A](t: Tree[A]): Int =
    foldTree(t)((0, true)):
        case ((_, true), _, (_, true)) => (1, false)          // los dos hijos vacíos: hoja
        case ((hl, _), _, (hr, _)) => (hl + hr, false)
    ._1

def maxTree(t: Tree[Int]): Option[Int] =
    foldTree(t)(None: Option[Int])((l, x, r) =>
        Some(List(l, Some(x), r).flatten.max))   // flatten quita los None

def mirror[A](t: Tree[A]): Tree[A] =
    foldTree[A, Tree[A]](t)(Empty())((l, x, r) => Node(r, x, l))

def contains[A](t: Tree[A], x: A): Boolean =
    foldTree(t)(false)((l, y, r) => l || y == x || r)
```

</details>

Siguiente: **08-abstraer-patrones-hof.ipynb**.